### CRISP-DM Phase 4.2 - Modeling : Country-level correlation

In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

In [ ]:
# Load the datasets
sensor = pd.read_csv('data/sensor.csv')
law = pd.read_csv('../law/data/law_classified.csv')

Distribution of EU countries

In [ ]:
def distribute_eu(df, members):
    eu_docs = df[df['Country'].isin(['EUR', 'EUE'])].copy()
    non_eu_docs = df[~df['Country'].isin(['EUR', 'EUE'])].copy()
    
    expanded = []
    for _, row in eu_docs.iterrows():
        doc_year = row['Year']
        for member, (entry_year, exit_year) in members.items():
            is_after_entry = doc_year >= entry_year
            is_before_exit = (exit_year is None) or (doc_year <= exit_year)
            
            if is_after_entry and is_before_exit:
                new_row = row.copy()
                new_row['Country'] = member
                new_row['eu_distributed'] = True
                expanded.append(new_row)
    
    eu_expanded = pd.DataFrame(expanded)
    result = pd.concat([non_eu_docs, eu_expanded], ignore_index=True)

    return result

In [ ]:
EU_YEARS_MEMBERS = {'BEL': (1958, None), 'FRA': (1958, None), 'DEU': (1958, None), 'ITA': (1958, None), 
                    'LUX': (1958, None), 'NLD': (1958, None), 'DNK': (1973, None), 'IRL': (1973, None), 
                    'GBR': (1973, 2020), 'GRC': (1981, None), 'PRT': (1986, None), 'ESP': (1986, None), 
                    'AUT': (1995, None), 'FIN': (1995, None), 'SWE': (1995, None), 'CYP': (2004, None), 
                    'CZE': (2004, None), 'EST': (2004, None), 'HUN': (2004, None), 'LVA': (2004, None), 
                    'LTU': (2004, None), 'MLT': (2004, None), 'POL': (2004, None), 'SVK': (2004, None), 
                    'SVN': (2004, None), 'BGR': (2007, None), 'ROU': (2007, None), 'HRV': (2013, None)}

law = distribute_eu(law, EU_YEARS_MEMBERS)

Compute relevant metrics

In [ ]:
## Legislative coverage
START = 2000
END = 2025
YEARS = list(range(START, END + 1))
HAZARDS = ['flood', 'drought', 'temperature_extremes', 'sea_level_rise', 'storm', 'wildfire',
          'melting', 'erosion', 'other', 'none']

law['Hazard'] = law['Hazard'].apply(lambda x: x if isinstance(x, list) else ast.literal_eval(x))
law_exploded = law.explode('Hazard').copy()

if 'eu_distributed' not in law_exploded.columns:
    law_exploded['eu_distributed'] = False

legislative_coverage = []

for country, df in law_exploded.groupby('Country', sort=True):
    country_name = df['Country_name'].iloc[0]
    
    for hazard in HAZARDS:
        subset = df[(df['Hazard'] == hazard) & (df['Year'] >= START) & (df['Year'] <= END)].sort_values('Year').copy()

        year_df = pd.DataFrame({'Year': YEARS})
        year_df['Country'] = country
        year_df['Country_name'] = country_name
        year_df['Hazard'] = hazard

        if subset.empty:
            year_df['Count'] = 0
            year_df['Count_national'] = 0
        else:
            yearly_counts = subset.groupby('Year').size().reset_index(name='new_laws')
            year_df = year_df.merge(yearly_counts, on='Year', how='left')
            year_df['new_laws'] = year_df['new_laws'].fillna(0)
            year_df['Count'] = year_df['new_laws'].cumsum()
            
            subset_national = subset[subset['eu_distributed'] != True]
            if subset_national.empty:
                year_df['Count_national'] = 0
            else:
                yearly_counts_nat = subset_national.groupby('Year').size().reset_index(name='new_laws_nat')
                year_df = year_df.merge(yearly_counts_nat, on='Year', how='left')
                year_df['new_laws_nat'] = year_df['new_laws_nat'].fillna(0)
                year_df['Count_national'] = year_df['new_laws_nat'].cumsum()

            # If last law appears before 2026
            last_year_with_law = subset['Year'].max()
            if last_year_with_law < END:
                final_count = year_df.loc[year_df['Year'] == last_year_with_law, 'Count'].values[0]
                year_df.loc[year_df['Year'] > last_year_with_law, 'Count'] = final_count
                
            if not subset_national.empty:
                last_year_nat = subset_national['Year'].max()
                if last_year_nat < END:
                    final_nat = year_df.loc[year_df['Year'] == last_year_nat, 'Count_national'].values[0]
                    year_df.loc[year_df['Year'] > last_year_nat, 'Count_national'] = final_nat


        legislative_coverage.append(year_df[['Country', 'Country_name', 'Year', 'Hazard', 'Count', 'Count_national']])

legislative_coverage = pd.concat(legislative_coverage, ignore_index=True)
legislative_coverage.to_csv(f'../law/outputs/legislative_coverage_country.csv', index=False)

In [ ]:
## Hazard intensity 
VARIABLES = ['2m_temperature', 'Instantaneous_wind_gust', 'Sea_level_anomaly', 
             'Snowmelt', 'SPEI', 'Total_precipitation']
BASELINE_END = 1999

sensor = sensor.groupby(['Country', 'Country_name', 'Year'])[VARIABLES].mean().reset_index()
hazard_intensity = sensor.copy()

for var in VARIABLES:
    if var == 'Sea_level_anomaly' or var == 'Snowmelt':
        # SPEI (reference period 1991–2020) and SLA (reference period 1993-2012) already usable as is
        continue
    
    baseline = sensor[sensor['Year'] <= BASELINE_END].groupby('Country')[var].mean().reset_index()
    baseline.columns = ['Country', f'{var}_mean'] 
    hazard_intensity = hazard_intensity.merge(baseline, on='Country', how='left')
    
    hazard_intensity[var] = ((hazard_intensity[var] - hazard_intensity[f'{var}_mean']))
    
    hazard_intensity = hazard_intensity.drop(columns=[f'{var}_mean'])
    
hazard_intensity = hazard_intensity[(hazard_intensity['Year'] >= 1995) & (hazard_intensity['Year'] <= 2025)].copy()
hazard_intensity.to_csv(f'outputs/hazard_intensity_country.csv', index=False)

Correlation

In [ ]:
hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

sensor_countries = set(hazard_intensity['Country'].dropna().unique())
law_countries = set(legislative_coverage['Country'].dropna().unique())
common_countries = sensor_countries & law_countries
country_name = hazard_intensity.drop_duplicates(subset=['Country']).set_index('Country')['Country_name'].to_dict()

correlation = []

for hazard, variable in hazard_variable_dict.items():
    for country in common_countries:
        country_intensity = hazard_intensity[hazard_intensity['Country'] == country]
        country_coverage = legislative_coverage[(legislative_coverage['Country'] == country) & (legislative_coverage['Hazard'] == hazard)]

        data = pd.merge(country_intensity[['Year', variable]], country_coverage[['Year', 'Count']], 
                        on='Year', how='inner').dropna()

        if data[variable].nunique() == 1 or data['Count'].nunique() == 1:  
            continue          
        try:
            rho, p = spearmanr(data[variable], data['Count'])
            correlation.append({'Hazard': hazard, 'Country': country, 'Country_name': country_name.get(country), 
                                'Rho': rho, 'p_value': p}) 
        except Exception as e:
            continue

correlation = pd.DataFrame(correlation)
correlation.to_csv(f'outputs/correlation_country.csv', index=False)

Legislation lag

In [ ]:
lag_results = []

for lag in [0, 1, 2, 3, 4, 5]:
    lagged_coverage = legislative_coverage.copy()
    lagged_coverage['Year'] = lagged_coverage['Year'] - lag

    lagged_correlation = []

    for hazard, variable in hazard_variable_dict.items():
        for country in common_countries:
            country_intensity = hazard_intensity[hazard_intensity['Country'] == country]
            country_coverage = lagged_coverage[(lagged_coverage['Country'] == country) & (lagged_coverage['Hazard'] == hazard)]

            data = pd.merge(country_intensity[['Year', variable]], country_coverage[['Year', 'Count']],
                            on='Year', how='inner').dropna()

            if data[variable].nunique() == 1 or data['Count'].nunique() == 1:
                continue
            try:
                rho, p = spearmanr(data[variable], data['Count'])
                lagged_correlation.append({'Hazard': hazard, 'Country': country, 'Rho': rho, 'p_value': p})
            except Exception as e:
                continue

    lagged_correlation = pd.DataFrame(lagged_correlation)
    sig_count = (lagged_correlation['p_value'] < 0.05).sum() if len(lagged_correlation) else 0
    total = len(lagged_correlation)

    lag_results.append({'lag': lag, 'significant': sig_count, 'total': total,
                        'pct': sig_count/total*100 if total > 0 else 0})

lag_results = pd.DataFrame(lag_results)
print(lag_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lag_results['lag'], lag_results['pct'], marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Years lagged')
ax.set_ylabel('Percentage of significant correlations')
ax.set_title('Effect of lag on correlation significance')
ax.set_xticks(lag_results['lag'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/4_legislation_lag_country.png', dpi=150, bbox_inches='tight')
plt.close()